In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

In [75]:
from openai import OpenAI
import os

from openai import OpenAI
openai_client = OpenAI()

In [76]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [77]:
llm("Hey, what's up?")

'Hey! Not much—just here and ready to help. What’s up with you?'

In [78]:
question = "I just discovered the course now. Can I join now ?"
answer=llm(question)
print(answer)

Yes — in many cases you can still join, but it depends on the course’s enrollment rules and whether registration is still open.

A good reply you can send is:

**“I just discovered the course now. Is it still possible for me to join?”**

If you want, I can also help you make it:
- more polite,
- more formal,
- or shorter for email/text.


In [79]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [80]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

answer = llm(prompt)
print(answer)

Yes, you can join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [81]:
# RAG stands for Retrieval-Augmented Generation
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [82]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f'''{url_prefix}/{course["path"]}'''

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

print(documents[0])
len(documents)

{'id': '9e508f2212', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: When does the course start?', 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}


1350

In [83]:
from minsearch import Index


index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [84]:
question = "I just discovered the course. Can I join now?"

In [85]:
def search(question, course="llm-zoomcamp"):
    boost_dict={"question": 2.0, "section": 0.5}
    filter_dict={"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

search_results = search(question)

In [86]:
def build_context(search_results):
    lines=[]

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: "+ doc["question"])
        lines.append("A: "+ doc["answer"])
        lines.append("")
    
    return "\n".join(lines).strip()

In [87]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [88]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [89]:
question = "I just discovered the course. Can I join now?"

prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [90]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [91]:
response.output_text

response.usage

ResponseUsage(input_tokens=480, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=44, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=524)

In [92]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.000558

In [93]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [97]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history=[{"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
        
    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [98]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(INSTRUCTIONS, prompt, model=model)

In [99]:
answer = rag("I just discovered the course. Can I join now?")

print(answer)

Yes, you can still join now. If you want a certificate, make sure to submit your project while submissions are still open.


In [100]:
rag("How do I get a certificate?")

'You can get a certificate only if you finish the course with a **live cohort** and **pass the Capstone project**.\n\nA few important notes:\n- **Self-paced mode does not provide certificates.**\n- You also need to **peer-review 3 capstones** after submitting your project.\n- Homework is **not mandatory** for the certificate, though it is recommended.\n\nIf you want your real name on the certificate, set your **official name** in your course profile.'

In [104]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

# response = openai_client.responses.create(
#     model="gpt-5.4-mini",
#     input=messages,
# )

# response.output_text

In [105]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [106]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [108]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output[0]

ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment late join open enrollment"}', call_id='call_V4pHsKUkidhIPPj43bN26IBI', name='search', type='function_call', id='fc_08ad4d83f88fcc6e006a3966b1656c819bbccdd0abbfe05cbe', namespace=None, status='completed')

In [109]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [110]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [111]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want to receive a certificate, though, you need to submit your project while the course is still accepting submissions.'

In [112]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(734, 37)

In [114]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(usage.input_tokens, usage.output_tokens)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001323


In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

"You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."

In [117]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [119]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course late enrollment discovered course can I join"}
function_call: search {"query":"course discovered can I join enrollment signup late join"}


In [120]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join. The course materials are available, and you can start learning and submitting homework while the submission forms are open.

One important caveat: if you want a certificate, you need to complete the project and submit it while the course is still accepting submissions. Certificates are not available for fully self-paced completion.

If you want, I can also help you find the course docs, GitHub repo, or explain how to get started.


In [121]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [122]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama local run install setup local model run course FAQ"}
function_call: search {"query":"Olama locally run local installation model command FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - **macOS**: download the `.pkg` from [ollama.com/download](https://ollama.com/download) and install it.
   - **Windows**: download the `.msi` from the same page and install it.
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model and open a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from the Ollama server.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```
   Then:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"r

'To run Ollama locally:\n\n1. **Install Ollama**\n   - **macOS**: download the `.pkg` from [ollama.com/download](https://ollama.com/download) and install it.\n   - **Windows**: download the `.msi` from the same page and install it.\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and open a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from the Ollama server.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n   Then:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection refused error, restarting the server often helps:\n```bash\nnohu

In [123]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment can I still join discovered course"}
function_call: search {"query":"course start date enrollment late join FAQ"}
function_call: search {"query":"can I still join the course if I discovered it late FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. You can also start learning and follow the materials even if you discovered the course late.

If you want, I can also help with:
- how to start from scratch,
- whether homework is required,
- or how certificates work.


'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open. You can also start learning and follow the materials even if you discovered the course late.\n\nIf you want, I can also help with:\n- how to start from scratch,\n- whether homework is required,\n- or how certificates work.'

In [124]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course after it started enrollment late registration discovered course can I join"}
iteration #2...
function_call: search {"query":"certificate self-paced live cohort peer-review project accepting submissions join after discovered course"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

A couple of important notes:
- You can start learning and follow the materials even if you discovered it late.
- If you want a certificate, you need to submit your project while submissions are still open.
- Certificates are only awarded for the live cohort, not for fully self-paced participation.

If you want, I can also explain how registration, homework, and the certificate process work.


'Yes — you can still join the course.\n\nA couple of important notes:\n- You can start learning and follow the materials even if you discovered it late.\n- If you want a certificate, you need to submit your project while submissions are still open.\n- Certificates are only awarded for the live cohort, not for fully self-paced participation.\n\nIf you want, I can also explain how registration, homework, and the certificate process work.'

In [125]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit course FAQ chess opening"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry about “queen gambit,” and it looks like that isn’t a course/logistics question.

If you meant something course-related, feel free to rephrase and I’ll look it up. Are there other areas you want to explore?


'I couldn’t find any course FAQ entry about “queen gambit,” and it looks like that isn’t a course/logistics question.\n\nIf you meant something course-related, feel free to rephrase and I’ll look it up. Are there other areas you want to explore?'

In [126]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [127]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [128]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [129]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [131]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [132]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [133]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [136]:
result.cost

CostInfo(input_cost=Decimal('0.00265425'), output_cost=Decimal('0.0015165'), total_cost=Decimal('0.00417075'))

In [135]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local setup install Ollama course FAQ"}', ca

In [137]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received
